# 01 Logistic Regression: Understanding Prediction Probability with Heart-Disease Data

This notebook uses `heart_data.csv`. Each row represents one person's body measurements, such as age, height, weight, blood pressure, cholesterol, and glucose.

The modeling question is: **given these measurements, predict whether `cardio` is 0 or 1.**

- `cardio = 0`: the record has no cardiovascular disease label
- `cardio = 1`: the record has a cardiovascular disease label

A key feature of Logistic Regression is that it predicts a class and also returns the probability of belonging to class 1.


## Before Running

These notebooks are teaching demos, not medical diagnoses or biology conclusions. They use real tabular data from the project, but the purpose is to learn how machine-learning models work.

Data paths:

- Heart-disease data: `../data/ml_data/heart_data.csv`
- Penguin data: `../data/ml_data/penguins.csv`

The recommended environment is the existing Docker/Jupyter setup in this project, because `environment/Dockerfile` already installs `numpy`, `pandas`, `scikit-learn`, `matplotlib`, and `seaborn`.

For a local Python environment, install:

```bash
pip install numpy pandas scikit-learn matplotlib seaborn notebook
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. Load and Inspect the Data

First read the CSV and look at the first few rows. The first step in machine learning is usually not model training, but understanding what each column means and which column is the target.


In [ ]:
candidate_paths = [
    Path('../../data/ml_data/heart_data.csv'),
    Path('../data/ml_data/heart_data.csv'),
    Path('/workspace/data/ml_data/heart_data.csv'),
    Path('/Users/nancy/Desktop/cancer-predictor/data/ml_data/heart_data.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Cannot find heart_data.csv. Please check data/ml_data/heart_data.csv')

heart = pd.read_csv(data_path)

print('Data path:', data_path)
print('Data shape:', heart.shape)
heart.head()


In [ ]:
print('cardio label distribution:')
print(heart['cardio'].value_counts().sort_index())

plt.figure(figsize=(5, 4))
sns.countplot(data=heart, x='cardio', hue='cardio', palette=['#2b7bba', '#d95f02'], legend=False)
plt.title('Target label distribution: cardio')
plt.xlabel('cardio: 0=no record, 1=recorded disease')
plt.ylabel('Count')
plt.show()


## 2. Light Cleaning and Feature Setup

The raw `age` column is stored as days, which is not intuitive, so we convert it to `age_years`.

We also compute BMI:

`BMI = weight kg / height m^2`

Real-world data may contain entry errors, such as unrealistic blood-pressure values. For this teaching demo, we keep only a reasonable range.


In [ ]:
heart = heart.copy()
heart['age_years'] = heart['age'] / 365.25
heart['bmi'] = heart['weight'] / (heart['height'] / 100) ** 2

clean = heart.query(
    '120 <= height <= 220 and 35 <= weight <= 200 and 80 <= ap_hi <= 220 and 40 <= ap_lo <= 140 and 10 <= bmi <= 60'
).copy()

print('Rows before cleaning:', len(heart))
print('Rows after cleaning:', len(clean))
clean[['age_years', 'height', 'weight', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'cardio']].head()


## 3. Data Analysis: Understand the Data with Charts

Following the Data Analysis workflow, we first define what these charts answer:

- **Analysis question**: which variables are more related to `cardio=1`, and are there clear outliers or group differences?
- **Comparison groups**: people with `cardio=0` versus people with `cardio=1`.
- **Sample definition**: the cleaned `clean` dataset from the previous step.
- **Limitations**: these charts show correlation, not proof that any factor causes heart disease.

The chart titles, axes, and legends use English to avoid font issues in Jupyter.


In [ ]:
plot_sample = clean.sample(n=2500, random_state=RANDOM_STATE).copy()
plot_sample['cardio_label'] = plot_sample['cardio'].map({
    0: 'cardio=0: no record',
    1: 'cardio=1: recorded disease',
})

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=plot_sample,
    x='age_years',
    y='ap_hi',
    hue='cardio_label',
    hue_order=['cardio=0: no record', 'cardio=1: recorded disease'],
    palette={
        'cardio=0: no record': '#2b7bba',
        'cardio=1: recorded disease': '#d95f02',
    },
    alpha=0.45,
    s=35,
)
plt.title('Age and systolic blood pressure by cardio label')
plt.xlabel('Age in years')
plt.ylabel('Systolic blood pressure: ap_hi')
plt.legend(title='Meaning of point color')
plt.show()


### 3.1 Distribution Differences in Age, Blood Pressure, and BMI

These charts compare age, blood pressure, and BMI between the `cardio=0` and `cardio=1` groups.

Reading tip: if the two distributions overlap heavily, one variable alone is unlikely to predict well. If one group shifts right or has a higher boxplot, that variable may help the model.


In [ ]:
analysis_data = clean.sample(n=min(8000, len(clean)), random_state=RANDOM_STATE).copy()
analysis_data['cardio_label'] = analysis_data['cardio'].map({
    0: 'cardio=0: no record',
    1: 'cardio=1: recorded disease',
})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(
    data=analysis_data,
    x='age_years',
    hue='cardio_label',
    bins=25,
    stat='density',
    common_norm=False,
    alpha=0.45,
    palette=['#2b7bba', '#d95f02'],
    ax=axes[0],
)
axes[0].set_title('Age distribution by cardio label')
axes[0].set_xlabel('Age in years')
axes[0].set_ylabel('Density')

sns.boxplot(
    data=analysis_data,
    x='cardio_label',
    y='ap_hi',
    hue='cardio_label',
    palette=['#2b7bba', '#d95f02'],
    legend=False,
    ax=axes[1],
)
axes[1].set_title('Systolic blood pressure by cardio label')
axes[1].set_xlabel('Cardio label')
axes[1].set_ylabel('ap_hi')
axes[1].tick_params(axis='x', rotation=15)

sns.boxplot(
    data=analysis_data,
    x='cardio_label',
    y='bmi',
    hue='cardio_label',
    palette=['#2b7bba', '#d95f02'],
    legend=False,
    ax=axes[2],
)
axes[2].set_title('BMI by cardio label')
axes[2].set_xlabel('Cardio label')
axes[2].set_ylabel('BMI')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


### 3.2 Grouped Cardio Rate

Next we inspect human-readable groups: cholesterol level, glucose level, lifestyle flags, and blood-pressure stage.

`cardio rate` is the proportion of `cardio=1` within each group. A higher rate means that group more often has `cardio=1` in this dataset.


In [ ]:
rate_data = clean.copy()
rate_data['bp_stage'] = pd.cut(
    rate_data['ap_hi'],
    bins=[0, 120, 140, 1000],
    labels=['ap_hi < 120', '120 <= ap_hi < 140', 'ap_hi >= 140'],
)

chol_rate = rate_data.groupby('cholesterol', as_index=False).agg(
    cardio_rate=('cardio', 'mean'),
    count=('cardio', 'size'),
)
gluc_rate = rate_data.groupby('gluc', as_index=False).agg(
    cardio_rate=('cardio', 'mean'),
    count=('cardio', 'size'),
)
bp_rate = rate_data.groupby('bp_stage', observed=True, as_index=False).agg(
    cardio_rate=('cardio', 'mean'),
    count=('cardio', 'size'),
)

lifestyle_rows = []
for column in ['smoke', 'alco', 'active']:
    grouped = rate_data.groupby(column)['cardio'].mean().reset_index()
    for _, row in grouped.iterrows():
        lifestyle_rows.append({
            'feature': column,
            'value': int(row[column]),
            'cardio_rate': row['cardio'],
        })
lifestyle_rate = pd.DataFrame(lifestyle_rows)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.barplot(data=chol_rate, x='cholesterol', y='cardio_rate', hue='cholesterol', palette='Blues', legend=False, ax=axes[0, 0])
axes[0, 0].set_title('Cardio rate by cholesterol level')
axes[0, 0].set_xlabel('Cholesterol level')
axes[0, 0].set_ylabel('Cardio rate')
axes[0, 0].set_ylim(0, 1)

sns.barplot(data=gluc_rate, x='gluc', y='cardio_rate', hue='gluc', palette='Oranges', legend=False, ax=axes[0, 1])
axes[0, 1].set_title('Cardio rate by glucose level')
axes[0, 1].set_xlabel('Glucose level')
axes[0, 1].set_ylabel('Cardio rate')
axes[0, 1].set_ylim(0, 1)

sns.barplot(data=bp_rate, x='bp_stage', y='cardio_rate', hue='bp_stage', palette='Reds', legend=False, ax=axes[1, 0])
axes[1, 0].set_title('Cardio rate by systolic BP stage')
axes[1, 0].set_xlabel('Systolic BP stage')
axes[1, 0].set_ylabel('Cardio rate')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].tick_params(axis='x', rotation=15)

sns.barplot(data=lifestyle_rate, x='feature', y='cardio_rate', hue='value', palette='Set2', ax=axes[1, 1])
axes[1, 1].set_title('Cardio rate by lifestyle indicators')
axes[1, 1].set_xlabel('Feature')
axes[1, 1].set_ylabel('Cardio rate')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].legend(title='Value')

plt.tight_layout()
plt.show()


### 3.3 Correlation Heatmap

The correlation heatmap helps us quickly see which numeric columns move together with `cardio`.

Correlation is not causation. It only suggests which variables the model may learn useful information from.


In [ ]:
corr_columns = [
    'age_years',
    'bmi',
    'ap_hi',
    'ap_lo',
    'cholesterol',
    'gluc',
    'smoke',
    'alco',
    'active',
    'cardio',
]

corr = clean[corr_columns].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, square=True)
plt.title('Correlation heatmap for heart prediction features')
plt.show()

corr['cardio'].sort_values(ascending=False).to_frame('correlation_with_cardio').round(3)


## 4. Train Logistic Regression

We select easy-to-understand features:

- `age_years`: age
- `bmi`: BMI
- `ap_hi`, `ap_lo`: blood pressure
- `cholesterol`, `gluc`: cholesterol and glucose levels
- `smoke`, `alco`, `active`: smoking, alcohol, and activity flags

Logistic Regression learns the direction and strength of each signal, then outputs a probability between 0 and 1.


In [ ]:
# These columns are the model inputs, also called features.
features = ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

# This column is the answer the model learns to predict, also called the target.
target = 'cardio'

# Sample 12,000 rows so the notebook runs faster while staying large enough for a demo.
model_data = clean[features + [target]].sample(n=12000, random_state=RANDOM_STATE)

# X stores only input features such as age, BMI, and blood pressure.
X = model_data[features]

# y stores the correct label: whether cardio is 0 or 1.
y = model_data[target]

# train_test_split divides the data into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    # X contains all input features, and y contains the matching labels.
    X, y,
    # test_size=0.25 keeps 25% of the data for testing and 75% for training.
    test_size=0.25,
    # random_state fixes the split so results are reproducible.
    random_state=RANDOM_STATE,
    # stratify=y keeps the cardio=0/1 proportions similar in train and test sets.
    stratify=y
)

# make_pipeline connects steps: scale first, then train Logistic Regression.
model = make_pipeline(
    # StandardScaler puts different numeric units on similar scales.
    StandardScaler(),
    # LogisticRegression is the prediction model; max_iter gives it enough iterations, and random_state supports reproducibility.
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
)

# fit means training: the model learns patterns from X_train and y_train.
model.fit(X_train, y_train)

print('Training complete')
print('Test accuracy:', round(model.score(X_test, y_test), 3))


## 5. Predict One Example Person

The person below is **not copied from the original CSV**. It is a hand-written example used only to show how to use the model.

The model outputs:

- predicted class: 0 or 1
- predicted probability: whether the example looks more like class 0 or class 1

Keep two numbers separate:

- **Test accuracy**: how often the model is correct across many test samples. For example, `0.74` means roughly 74% correct.
- **One person's predicted probability**: how strongly the model leans toward `cardio=1` or `cardio=0` for one input row.

So if `0.74` appears as test accuracy, it is the model's overall exam score, not one person's risk probability.


In [ ]:
# This is a hand-written example person, not a row copied from the CSV.
new_person = pd.DataFrame({
    'age_years': [55],
    'bmi': [28],
    'ap_hi': [145],
    'ap_lo': [90],
    'cholesterol': [2],
    'gluc': [1],
    'smoke': [0],
    'alco': [0],
    'active': [1],
})

predicted_label = model.predict(new_person)[0]
predicted_proba = model.predict_proba(new_person)[0]

same_as_existing_row = (clean[features].round(6) == new_person.iloc[0].round(6)).all(axis=1).any()

risk_probability = predicted_proba[1]
if risk_probability >= 0.80:
    probability_level = 'high'
elif risk_probability >= 0.60:
    probability_level = 'medium-high'
elif risk_probability >= 0.40:
    probability_level = 'uncertain / near the middle'
else:
    probability_level = 'low'

display(new_person)
print('Is this exact person copied from the CSV data?', same_as_existing_row)
print('Predicted cardio:', predicted_label)
print('Probability for cardio=0:', round(predicted_proba[0], 3))
print('Probability for cardio=1:', round(predicted_proba[1], 3))
print('How to read the cardio=1 probability:', probability_level)


## 6. Evaluation: How Well Did the Model Predict?

Accuracy is the proportion of test samples predicted correctly. If test accuracy is about `0.74`, the model predicts roughly 74 out of 100 test samples correctly.

How should we read that score?

- For an introductory demo, `0.74` means the model learned some real signal and is better than random guessing.
- In this dataset, `cardio=0` and `cardio=1` are fairly balanced, so random guessing would be around `0.50`.
- For health or medical prediction, `0.74` is not high enough for serious diagnosis. It is only appropriate for learning the modeling workflow.

To improve test accuracy, focus on making the model genuinely better rather than pushing one example's probability higher. Common options include cleaning unrealistic blood pressure, height, and weight values; adding more useful features; trying non-linear models such as Random Forest; tuning parameters; and always confirming improvements on a test set or through cross-validation.

A confusion matrix gives a more detailed view of which 0 labels were predicted as 1 and which 1 labels were predicted as 0.


In [ ]:
y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print()
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=['cardio=0', 'cardio=1']))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['cardio=0', 'cardio=1'])
disp.plot(cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.show()


## 7. Improving Test Accuracy: Parameter Tuning

Parameter tuning means keeping the same model type, Logistic Regression, while trying several settings to see which one validates best.

We tune three common parameters:

- `C`: controls how flexible the model is. Smaller `C` is more conservative; larger `C` fits the training data more strongly.
- `class_weight`: lets the model pay more attention to class balance if one class is harder to predict.
- `solver`: the optimization method the model uses to find parameters.

Important reminder: tuning does not guarantee higher accuracy. It is a systematic way to try settings. The final judgment should use test-set performance, not only training performance.


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Record baseline test accuracy for comparison.
baseline_accuracy = model.score(X_test, y_test)

# This pipeline is the same as before: scale first, then train Logistic Regression.
tuning_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
)

# param_grid lists the parameter combinations to try.
# The logisticregression__ prefix means this parameter belongs to the LogisticRegression step in the pipeline.
param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'logisticregression__class_weight': [None, 'balanced'],
    'logisticregression__solver': ['lbfgs', 'liblinear'],
}

# Cross-validation splits the training set into 5 folds to avoid relying on a single split.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# GridSearchCV tries all parameter combinations and picks the one with the best average validation accuracy.
grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
tuned_accuracy = best_model.score(X_test, y_test)

print('Baseline test accuracy:', round(baseline_accuracy, 3))
print('Best CV accuracy during tuning:', round(grid_search.best_score_, 3))
print('Tuned test accuracy:', round(tuned_accuracy, 3))
print('Best parameters:')
print(grid_search.best_params_)


### How to Read the Tuning Result

- If `Tuned test accuracy` is higher than `Baseline test accuracy`, tuning helped on the test set.
- If the two scores are similar, the default settings were already good enough.
- If the tuned score is lower, tuning may have looked better in cross-validation without improving test performance.

For Logistic Regression, tuning usually gives limited gains. Other common improvement paths include:

- Add more informative features, such as pulse pressure `ap_hi - ap_lo`, age groups, or BMI groups.
- Handle unrealistic blood-pressure, height, and weight values more carefully.
- Try models that can learn non-linear relationships, such as Random Forest.
- Compare several models with cross-validation instead of relying on one train/test split.


## 8. Improving Test Accuracy: Feature Engineering

Feature engineering means creating new columns from domain knowledge and data meaning, instead of sending only the raw columns to the model.

For example, the raw data includes:

- `ap_hi`: systolic blood pressure
- `ap_lo`: diastolic blood pressure

We can create:

- `pulse_pressure = ap_hi - ap_lo`: pulse pressure
- `mean_arterial_pressure = (ap_hi + 2 * ap_lo) / 3`: an approximate mean arterial pressure

We can also add interaction features so a linear model can see that age and blood pressure increase together:

- `age_ap_hi = age_years * ap_hi`
- `age_bmi = age_years * bmi`

New columns are not guaranteed to help, so the final check should still use test accuracy.


In [ ]:
from sklearn.metrics import roc_auc_score

# Start from the same model_data so the comparison uses identical samples.
engineered_data = model_data.copy()

# Blood-pressure-related new features.
engineered_data['pulse_pressure'] = engineered_data['ap_hi'] - engineered_data['ap_lo']
engineered_data['mean_arterial_pressure'] = (engineered_data['ap_hi'] + 2 * engineered_data['ap_lo']) / 3

# Interaction features let a linear model see two indicators changing together.
engineered_data['age_bmi'] = engineered_data['age_years'] * engineered_data['bmi']
engineered_data['age_ap_hi'] = engineered_data['age_years'] * engineered_data['ap_hi']
engineered_data['bmi_ap_hi'] = engineered_data['bmi'] * engineered_data['ap_hi']

# Threshold features convert readable rules into 0/1 indicators.
engineered_data['high_cholesterol'] = (engineered_data['cholesterol'] >= 2).astype(int)
engineered_data['very_high_cholesterol'] = (engineered_data['cholesterol'] >= 3).astype(int)
engineered_data['high_gluc'] = (engineered_data['gluc'] >= 2).astype(int)
engineered_data['stage2_systolic'] = (engineered_data['ap_hi'] >= 140).astype(int)
engineered_data['stage2_diastolic'] = (engineered_data['ap_lo'] >= 90).astype(int)
engineered_data['older_than_50'] = (engineered_data['age_years'] >= 50).astype(int)
engineered_data['older_than_55'] = (engineered_data['age_years'] >= 55).astype(int)

engineered_features = features + [
    'pulse_pressure',
    'mean_arterial_pressure',
    'age_bmi',
    'age_ap_hi',
    'bmi_ap_hi',
    'high_cholesterol',
    'very_high_cholesterol',
    'high_gluc',
    'stage2_systolic',
    'stage2_diastolic',
    'older_than_50',
    'older_than_55',
]

X_eng = engineered_data[engineered_features]
y_eng = engineered_data[target]

X_eng_train, X_eng_test, y_eng_train, y_eng_test = train_test_split(
    X_eng, y_eng, test_size=0.25, random_state=RANDOM_STATE, stratify=y_eng
)

feature_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
)

feature_model.fit(X_eng_train, y_eng_train)

baseline_proba = model.predict_proba(X_test)[:, 1]
feature_proba = feature_model.predict_proba(X_eng_test)[:, 1]

baseline_accuracy = model.score(X_test, y_test)
feature_accuracy = feature_model.score(X_eng_test, y_eng_test)

comparison = pd.DataFrame({
    'model': ['baseline logistic regression', 'with feature engineering'],
    'test_accuracy': [baseline_accuracy, feature_accuracy],
    'roc_auc': [roc_auc_score(y_test, baseline_proba), roc_auc_score(y_eng_test, feature_proba)],
})

comparison


### Combine Feature Engineering with Tuning

The previous step added new features only. Now we combine those new features with `GridSearchCV`, so the model can see better inputs and automatically try several parameter settings.


In [ ]:
engineered_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
)

engineered_param_grid = {
    'logisticregression__C': [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
    'logisticregression__class_weight': [None, 'balanced'],
    'logisticregression__solver': ['lbfgs', 'liblinear'],
}

engineered_grid_search = GridSearchCV(
    estimator=engineered_pipeline,
    param_grid=engineered_param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
)

engineered_grid_search.fit(X_eng_train, y_eng_train)

best_engineered_model = engineered_grid_search.best_estimator_
engineered_tuned_accuracy = best_engineered_model.score(X_eng_test, y_eng_test)
engineered_tuned_proba = best_engineered_model.predict_proba(X_eng_test)[:, 1]
engineered_tuned_auc = roc_auc_score(y_eng_test, engineered_tuned_proba)

final_comparison = pd.DataFrame({
    'model': [
        'baseline logistic regression',
        'tuned baseline logistic regression',
        'feature engineering only',
        'feature engineering + tuning',
    ],
    'test_accuracy': [
        baseline_accuracy,
        tuned_accuracy,
        feature_accuracy,
        engineered_tuned_accuracy,
    ],
    'roc_auc': [
        roc_auc_score(y_test, baseline_proba),
        roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_eng_test, feature_proba),
        engineered_tuned_auc,
    ],
})

print('Best parameters after feature engineering:')
print(engineered_grid_search.best_params_)

display(final_comparison)


### What Does This Improvement Mean?

With this dataset and this split, feature engineering often gives a small improvement, but it will not usually move Logistic Regression from 0.74 to 0.90 all at once.

The reason is that Logistic Regression is still a linear model. New features can help it see more patterns, but if the real structure is complex, a model such as Random Forest may be more suitable.

A realistic learning takeaway is:

- Parameter tuning can fine-tune a model.
- Feature engineering is often more useful than parameter tuning alone.
- If the improvement is still small, the next step may be a different model or more informative data.


## 9. Inspect Feature Directions

Logistic Regression gives each feature a coefficient. A simple interpretation is:

- Positive coefficient: as the feature increases, the model leans more toward `cardio=1`.
- Negative coefficient: as the feature increases, the model leans more toward `cardio=0`.

This is a statistical pattern learned from this dataset, not a medical causal conclusion.


In [ ]:
log_reg = model.named_steps['logisticregression']
coef = pd.Series(log_reg.coef_[0], index=features).sort_values()

plt.figure(figsize=(7, 5))
sns.barplot(x=coef.values, y=coef.index, hue=coef.index, palette='vlag', legend=False)
plt.axvline(0, color='black', linewidth=1)
plt.title('Logistic Regression Feature Coefficients')
plt.xlabel('Coefficient: positive leans toward cardio=1, negative leans toward cardio=0')
plt.ylabel('Feature')
plt.show()

coef


## 10. Summary

Logistic Regression is useful when:

- You want class probabilities.
- You want a relatively interpretable model.
- The data pattern is close to a risk score built from adding several indicators.

In this example, the model uses age, BMI, blood pressure, and related information to compute a tendency score, then converts that score into the probability of `cardio=1`.
